# AGOL Demo | Load One Module File from GitHub

This notebook shows a simple pattern for loading one Python module file directly from a GitHub repository into an AGOL notebook session.

**What the helper function `load_github_module` does:**

1. Builds a raw GitHub URL for one `.py` file.
2. Downloads the file to a local cache in `/arcgis/home/modules_cache`.
3. Imports the file dynamically as a Python module.
4. Returns the imported module so you can call its functions.

*Why use this approach?*

- You can use one module file without installing the full package.
- You can pin `reference` to a tag or commit for reproducible scheduled runs.
- You can use `main` for fast development iteration.

In [9]:
import os
import getpass
from pathlib import Path
from dotenv import load_dotenv

### ArcGIS Online Login

You can log in to ArcGIS Online in three ways:

- **Environment variables**: 
  - add credentials to a `.env` file (see `notebooks/.env.example`), then call `get_agol_connection(use_pro=False)`.
- **ArcGIS Pro**: sign in through ArcGIS Pro, then call `get_agol_connection(use_pro=True)`.
- **Terminal input**: if the options above are not used or fail, you can enter username and password manually.

In [10]:
# --- Load environment variables from .env file ---
env_candidates = [
Path("notebooks") / ".env",
Path.cwd() / ".env",
]
env_file = next((p for p in env_candidates if p.exists()), None)

if env_file is not None:
  load_dotenv(env_file)
  print(f"Loaded variables from {env_file.resolve()}")
else:
  print("No .env file found. Set environment variables or use interactive prompt.")

Loaded variables from C:\Users\WILACA\git\miljodirektoratet\arcpy-utils-public\notebooks\.env


In [11]:
# --- Connect to ArcGIS Online ---
def get_agol_connection(use_pro: bool = False):
    """
    Return an authenticated GIS connection to ArcGIS Online.

    :param use_pro: If True, try ArcGIS Pro credentials first.
    :return: Authenticated GIS object.
    """
    
    from arcgis.gis import GIS
    
    if use_pro:
        try:
            gis = GIS("home")
            user = gis.users.me
            if user is not None:
                return gis
            print("Warning: ArcGIS Pro is connected, but no AGOL user was found.")
        except Exception as error:
            print(f"Could not connect to ArcGIS Pro credentials: {error}")

    print("Using environment variables or interactive prompt...")
    portal = os.getenv("AGOL_URL", "https://www.arcgis.com")
    username = os.getenv("AGOL_USERNAME") or input("AGOL username: ")
    password = os.getenv("AGOL_PASSWORD") or getpass.getpass("AGOL password: ")
    return GIS(portal, username, password)

gis = get_agol_connection(use_pro=False)  # Set to True to try ArcGIS Pro first
user = gis.users.me
if user is None:
    raise RuntimeError("Login failed: AGOL user information is unavailable.")

print(f">>> Logged in as: {user.username}")
print(f">>> AGOL organization / Portal: {gis.properties.get('urlKey')} ({gis.properties.get('name')})")

Using environment variables or interactive prompt...
>>> Logged in as: mdiradmin
>>> AGOL organization / Portal: miljodir-ekstern (Miljødirektoratet ekstern)


In [12]:
# --- Function: Load a specific Python module from GitHub ---
def load_github_module(
    module: str,
    owner: str,
    repo: str,
    reference: str | None = None,
):
    """
    Load one Python module file from a GitHub repository into the current AGOL session.

    `reference` can be a branch, tag, or commit hash.

    :param module: Path to module file in repository, for example "modules/hello_world.py".
    :param owner: GitHub owner or organization.
    :param repo: GitHub repository name.
    :param reference: Branch, tag, or commit hash. Defaults to "main".
    :return: Imported module object.
    """

    import importlib.util
    import urllib.request
    
    reference = reference or "main"
    local_file = f"/arcgis/home/modules_cache/{reference[:7]}/{module}"
    os.makedirs(os.path.dirname(local_file), exist_ok=True)

    url = f"https://raw.githubusercontent.com/{owner}/{repo}/{reference}/{module}"
    if not os.path.exists(local_file):
        urllib.request.urlretrieve(url, local_file)

    module_stub = module.removesuffix(".py")
    module_name = f"{module_stub}_{reference[:7]}"

    spec = importlib.util.spec_from_file_location(module_name, local_file)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not create import spec for {local_file}")

    loaded_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(loaded_module)
    return loaded_module


### Example A: load module `hello.py` by version tag

- reproducible
- recommended for stable releases


In [13]:
# Example A: load module `hello.py` by version tag
main = load_github_module(
    owner="miljodirektoratet",
    repo="gis-utils-public",
    reference="v0.0.4",
    module="src/gis_utils_public/main.py",
)
main.main()

Hello from gis-utils-public!
Version: unknown
Version metadata not available in this runtime.


In [16]:
# Example B: load module `hello.py` by commit hash (reproducible)
main = load_github_module(
    owner="miljodirektoratet",
    repo="gis-utils-public",
    reference="163e0cea6c0178b6483da7bd8ce63bf75adb1422",
    module="src/gis_utils_public/main.py",
)
main.main()

Hello from gis-utils-public!
Version: unknown
Version metadata not available in this runtime.


In [ ]:
# Example C: load module `hello.py` by branch name (not reproducible, only recommended for active development not for notebook-jobs)
main = load_github_module(
    owner="miljodirektoratet",
    repo="gis-utils-public",
    reference="main",
    module="src/gis_utils_public/main.py",
)

main.main()

Hello from gis-utils-public!
Version: unknown
Version metadata not available in this runtime.


In [20]:
# Example D: load the package `arcgis_utils_public` (not recommended, costs more credits)
%pip install --no-cache-dir "https://github.com/miljodirektoratet/gis-utils-public/archive/refs/tags/v0.0.4.zip"

     - 0 bytes ? 0:00:00
     - 38.0 kB 1.8 MB/s 0:00:00
     - 48.6 kB 817.2 kB/s 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for gis-utils-public: filename=gis_utils_public-0.0.3-py3-none-any.whl size=9259 sha256=b4ede01e4280a5b641f34316e1b9be9a6ae01e82a7dc41dd1b8b3d43fa54e0b8
  Stored in directory: C:\Users\WILACA\AppData\Local\Temp\pip-ephem-wheel-cache-j7u26vrm\wheels\80\cf\8a\b20401c922f1ff5e078745c13668c3b4ab2a69f14614f47573
Successfully built gis-utils-public
Note: you may need to restart the kernel to use updated packages.


In [21]:
from gis_utils_public import arcgis_utils
